# Final refit and single held-out test evaluation

This notebook loads completed Optuna studies, selects winners using CV results only, trains the models on the persisted train/validation split, and performs a single held-out test evaluation.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import copy
import os
import pickle
import random

import numpy as np
import optuna
import pandas as pd
import torch
import yaml
from dotenv import load_dotenv
from sklearn import set_config

from my_project.data.utils import load_column_config, select_modeling_features
from my_project.experiments import medical as medical_cfg
from my_project.gnn.models import MyGNN, MyMLP
from my_project.graphs import build_base_graphs, build_ii_graphs, build_true_graph
from my_project.information_theory import compute_info
from my_project.training.final_training import fit_final_gnn, fit_final_xgboost
from my_project.training.training_helpers import _build_study_name

load_dotenv(override=True)
set_config(transform_output="pandas")
torch.set_float32_matmul_precision("high")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
CUDA_AVAILABLE = torch.cuda.is_available()

## Configuration and data

In [ ]:
EXPERIMENT_NAME = "medical_9"

DATA_TYPE = "medical"
INTERACTION_THRESHOLDS = [0.95, 0.9, 0.8]
PERMUTE_SEEDS = list(range(10))
MAX_EPOCHS = 200
SAVE_TEST_PREDICTIONS = True

CONFIGS_DIR = os.environ["CONFIGS_DIR"]
DATA_DIR = os.environ["DATA_DIR"]
RESULTS_DIR = os.environ["RESULTS_DIR"]
OPTUNA_STORAGE_URL = os.environ["OPTUNA_STORAGE_URL"]
MLFLOW_TRACKING_URI = os.environ["MLFLOW_TRACKING_URI"]

with open(os.path.join(CONFIGS_DIR, "data.yaml"), encoding="utf-8") as f:
    data_config = yaml.safe_load(f)

col_cfg = load_column_config(CONFIGS_DIR)
df = pd.read_pickle(
    os.path.join(DATA_DIR, data_config["paths"]["preprocessed_data_path"])
)
y = df[col_cfg.TARGET]
X = select_modeling_features(df, CONFIGS_DIR)

In [ ]:
with open(
    os.path.join(DATA_DIR, data_config["paths"]["crossval_indices_path"]), "rb"
) as f:
    folds = list(pickle.load(f))[:5]

with open(
    os.path.join(DATA_DIR, data_config["paths"]["split_indices_path"]), "rb"
) as f:
    final_split = pickle.load(f)

FINAL_TRAIN_INDEX = pd.Index(final_split["train"])
FINAL_VALID_INDEX = pd.Index(final_split["valid"])
FINAL_TRAIN_VALID_INDEX = pd.Index(final_split["train_valid"])
TEST_INDEX = pd.Index(final_split["test"])

assert not set(FINAL_TRAIN_INDEX) & set(FINAL_VALID_INDEX)
assert not set(FINAL_TRAIN_VALID_INDEX) & set(TEST_INDEX)
assert set(FINAL_TRAIN_INDEX) | set(FINAL_VALID_INDEX) == set(FINAL_TRAIN_VALID_INDEX)
assert set(FINAL_TRAIN_VALID_INDEX) == set(folds[0]["train"]) | set(folds[0]["valid"])
assert set(TEST_INDEX) == set(folds[0]["test"])

pd.Series(
    {
        "train": len(FINAL_TRAIN_INDEX),
        "valid": len(FINAL_VALID_INDEX),
        "train_valid": len(FINAL_TRAIN_VALID_INDEX),
        "test": len(TEST_INDEX),
    }
)

## CV winners from Optuna

In [ ]:
BASE_GRAPH_NAMES = ["empty", "fully_connected"]
INTERACTION_GRAPH_NAMES = [
    f"ii_graph_{int(threshold * 100)}" for threshold in INTERACTION_THRESHOLDS
]
PERMUTED_GRAPH_NAMES = [
    f"{graph_name}_permuted_{seed}"
    for graph_name in INTERACTION_GRAPH_NAMES
    for seed in PERMUTE_SEEDS
]
ALL_GNN_GRAPH_NAMES = [
    *BASE_GRAPH_NAMES,
    *INTERACTION_GRAPH_NAMES,
    *PERMUTED_GRAPH_NAMES,
]
CV_TRAIN_SIZE = len(folds[0]["train"])


def load_nn_result(model_name, graph_name, conv_layer=None):
    study_name = _build_study_name(
        EXPERIMENT_NAME,
        DATA_TYPE,
        model_name,
        conv_layer,
        graph_name,
        f"seed{SEED}",
        f"train{CV_TRAIN_SIZE}",
    )
    try:
        study = optuna.load_study(study_name=study_name, storage=OPTUNA_STORAGE_URL)
    except KeyError as exc:
        raise RuntimeError(f"Missing completed study: {study_name}") from exc
    best = study.best_trial
    return {
        "model_cls": model_name,
        "graph_name": graph_name,
        "study_name": study_name,
        "trial_number": best.number,
        "best_value": study.best_value,
        "best_params": best.user_attrs.get("params", dict(best.params)),
        "best_epochs": best.user_attrs.get("best_epochs", []),
    }


gnn_results = [
    load_nn_result("MyGNN", graph_name, conv_layer="GATConv")
    for graph_name in ALL_GNN_GRAPH_NAMES
]
mlp_result = load_nn_result("MyMLP", "empty")
FINAL_NN_RESULTS = [*gnn_results, mlp_result]

pd.DataFrame(FINAL_NN_RESULTS)[
    ["model_cls", "graph_name", "trial_number", "best_value", "study_name"]
]

In [ ]:
xgb_study_name = _build_study_name(EXPERIMENT_NAME, "XGBoost", "xgboost_full")
try:
    xgb_study = optuna.load_study(study_name=xgb_study_name, storage=OPTUNA_STORAGE_URL)
except KeyError as exc:
    raise RuntimeError(f"Missing completed study: {xgb_study_name}") from exc

xgb_trial = xgb_study.best_trial
xgb_result = {
    "model_cls": "XGBoost",
    "graph_name": "xgboost_full",
    "study_name": xgb_study_name,
    "trial_number": xgb_trial.number,
    "best_value": xgb_study.best_value,
    "best_params": xgb_trial.user_attrs.get("params", dict(xgb_trial.params)),
    "best_n_estimators": xgb_trial.user_attrs.get("best_n_estimators"),
}
if xgb_result["best_n_estimators"] is None:
    raise RuntimeError("Best XGBoost trial has no CV-selected best_n_estimators")
xgb_result

## Final graph built from the persisted training set

In [ ]:
GRAPH_PREPROCESSING_PIPELINE = copy.deepcopy(medical_cfg.GRAPH_PREPROCESSING_PIPELINE)
X_final_train = X.loc[FINAL_TRAIN_INDEX]
y_final_train = y.loc[FINAL_TRAIN_INDEX]

final_graph_pipeline = copy.deepcopy(GRAPH_PREPROCESSING_PIPELINE)
final_graph_input = final_graph_pipeline.fit_transform(X_final_train, y_final_train)
final_interaction = compute_info(final_graph_input, y_final_train, n_bins=None)

FINAL_GRAPHS = build_base_graphs(build_true_graph(final_interaction, []))
for threshold in INTERACTION_THRESHOLDS:
    graph_prefix = f"ii_graph_{int(threshold * 100)}"
    threshold_graphs = build_ii_graphs(
        final_interaction,
        permute_seeds=PERMUTE_SEEDS,
        threshold=threshold,
    )
    FINAL_GRAPHS[graph_prefix] = threshold_graphs["ii"]
    for seed in PERMUTE_SEEDS:
        FINAL_GRAPHS[f"{graph_prefix}_permuted_{seed}"] = threshold_graphs[
            f"ii_permuted_{seed}"
        ]

missing_graphs = {
    result["graph_name"] for result in FINAL_NN_RESULTS
} - FINAL_GRAPHS.keys()
if missing_graphs:
    raise KeyError(f"Missing final graphs: {sorted(missing_graphs)}")

## Refit GNN and MLP

In [ ]:
metric = "val/avg_precision"

In [ ]:
PREPROCESSING_PIPELINE = copy.deepcopy(medical_cfg.NN_PREPROCESSING_PIPELINE)
XGBOOST_PREPROCESSING_PIPELINE = copy.deepcopy(
    medical_cfg.XGBOOST_PREPROCESSING_PIPELINE
)
PRECISION = "16-mixed" if CUDA_AVAILABLE else "32-true"

trainer_kwargs = {
    "enable_progress_bar": False,
    "enable_model_summary": False,
    "log_every_n_steps": 5,
    "precision": PRECISION,
    "max_epochs": MAX_EPOCHS,
}
monitor_kwargs = {"monitor": metric, "mode": "max"}
early_stopping_kwargs = {
    "patience": 15,
    "verbose": False,
    "monitor": metric,
    "mode": "max",
}

FINAL_MODEL_DIR = os.path.join(
    RESULTS_DIR, "training", "gnn", "final_refit", EXPERIMENT_NAME
)
MODEL_CLASSES = {"MyGNN": MyGNN, "MyMLP": MyMLP}
final_results = []

In [ ]:
for result in FINAL_NN_RESULTS:
    if not result["best_epochs"]:
        raise RuntimeError(
            f"Best trial for {result['model_cls']}/{result['graph_name']} has no best_epochs"
        )
    final_results.append(
        fit_final_gnn(
            X=X,
            y=y,
            train_index=FINAL_TRAIN_INDEX,
            valid_index=FINAL_VALID_INDEX,
            test_index=TEST_INDEX,
            graph=FINAL_GRAPHS[result["graph_name"]],
            graph_name=result["graph_name"],
            model_cls=MODEL_CLASSES[result["model_cls"]],
            source_study_name=result["study_name"],
            source_trial_number=result["trial_number"],
            best_params=result["best_params"],
            best_epochs=result["best_epochs"],
            preprocessing_pipeline=PREPROCESSING_PIPELINE,
            trainer_kwargs=trainer_kwargs,
            monitor_kwargs=monitor_kwargs,
            early_stopping_kwargs=early_stopping_kwargs,
            tracking_uri=MLFLOW_TRACKING_URI,
            experiment_name=EXPERIMENT_NAME,
            output_dir=FINAL_MODEL_DIR,
            seed=SEED,
            keep_on_gpu=CUDA_AVAILABLE and result["graph_name"] != "fully_connected",
            save_predictions=SAVE_TEST_PREDICTIONS,
        )
    )

## Refit XGBoost

In [ ]:
final_results.append(
    fit_final_xgboost(
        X=X,
        y=y,
        train_index=FINAL_TRAIN_INDEX,
        valid_index=FINAL_VALID_INDEX,
        test_index=TEST_INDEX,
        source_study_name=xgb_result["study_name"],
        source_trial_number=xgb_result["trial_number"],
        best_params=xgb_result["best_params"],
        best_n_estimators=xgb_result["best_n_estimators"],
        preprocessing_pipeline=XGBOOST_PREPROCESSING_PIPELINE,
        tracking_uri=MLFLOW_TRACKING_URI,
        experiment_name=EXPERIMENT_NAME,
        output_dir=FINAL_MODEL_DIR,
        save_predictions=SAVE_TEST_PREDICTIONS,
        early_stopping_rounds=15,
    )
)

## Held-out test results

In [ ]:
FINAL_TEST_RESULTS = pd.DataFrame(final_results)
metric_names = [
    "n_samples",
    "auc",
    "avg_precision",
    "balanced_accuracy",
    "f1",
    "precision",
    "recall",
    "specificity",
    "accuracy",
]
final_columns = [
    "model_cls",
    "graph_name",
    "final/epochs_trained",
    "final/global_steps",
    "final/best_monitor_score",
    "final/n_estimators",
    *[
        f"{split}/{metric_name}"
        for split in ("train", "valid", "train_valid", "test")
        for metric_name in metric_names
    ],
    "reused",
    "run_id",
    "model_path",
    "predictions_path",
]
FINAL_TEST_RESULTS[
    [column for column in final_columns if column in FINAL_TEST_RESULTS.columns]
]